In [1]:
import pandas as pd, numpy as np
import xgboost as xgb
import shap
print("Environment OK")

Environment OK


In [ ]:
import pandas as pd

df = pd.read_csv('../data/paysim.csv')
print(f"Shape: {df.shape}")

Shape: (6362620, 11)


In [ ]:
# Output: tên 11 cột, dtype từng cột, và bộ nhớ dùng. 
# Kết quả là bằng chứng cụ thể cho phần "Data Understanding" 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [ ]:
# Dữ liệu thật hình dung rõ hơn con số trong từng cột trông như thế nào
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0


In [6]:
# Kiểm tra giá trị null
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [7]:
# Kiểm tra duplicate
df.duplicated().sum()

np.int64(0)

In [8]:
# Thống kê mô tả cơ bản
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [9]:
# Kiểm tra giá trị duy nhất của các cột phân loại
df['type'].value_counts()
df['isFraud'].value_counts()
df['isFlaggedFraud'].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [10]:
# Verify ý nghĩa step
df['step'].min(), df['step'].max()
#Kỳ vọng: min = 1, max = 743 hoặc 744 → xác nhận đúng "1 step = 1 giờ, tổng ~30 ngày" (744 giờ = 31 ngày).

# Verify fraud chỉ xảy ra ở TRANSFER và CASH_OUT
df[df['isFraud'] == 1]['type'].value_counts()
# Đây là câu lệnh quan trọng nhất ngày 2 — nó tự tay kiểm chứng insight mà bạn đã ghi trong memory ("fraud chỉ xảy ra ở TRANSFER và CASH_OUT"). Nếu kết quả đúng như vậy (chỉ ra 2 loại này, không có PAYMENT/CASH_IN/DEBIT), bạn có bằng chứng chắc chắn để viết vào cả Data Understanding lẫn Discussion/Limitations sau này.

# Verify công thức số dư có khớp logic không
# Kiểm tra vài dòng xem oldbalanceOrg - amount có gần bằng newbalanceOrig không
df[['oldbalanceOrg', 'amount', 'newbalanceOrig']].head(10)
# Giúp bạn hiểu tại sao paper (và bạn ở tuần 1 ngày 4) cần tạo errorBalanceOrig — vì thực tế nhiều dòng số dư không khớp hoàn toàn với phép trừ đơn giản (có sai lệch), và bản thân độ lệch đó lại là tín hiệu hữu ích để phát hiện gian lận.

# Thử vài giao dịch fraud thật để có cảm giác trực quan
df[df['isFraud'] == 1].head(10)
# Đọc bằng mắt vài dòng — bạn sẽ thấy rõ pattern "rút cạn tài khoản" (newbalanceOrig gần như luôn = 0 sau giao dịch) mà không cần chờ đến bước EDA vẽ biểu đồ mới thấy.

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.00,C1305486145,181.00,0.0,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.0,C38997010,21182.0,0.00,1,0
251,1,TRANSFER,2806.00,C1420196421,2806.00,0.0,C972765878,0.0,0.00,1,0
252,1,CASH_OUT,2806.00,C2101527076,2806.00,0.0,C1007251739,26202.0,0.00,1,0
680,1,TRANSFER,20128.00,C137533655,20128.00,0.0,C1848415041,0.0,0.00,1,0
681,1,CASH_OUT,20128.00,C1118430673,20128.00,0.0,C339924917,6268.0,12145.85,1,0
724,1,CASH_OUT,416001.33,C749981943,0.00,0.0,C667346055,102.0,9291619.62,1,0
969,1,TRANSFER,1277212.77,C1334405552,1277212.77,0.0,C431687661,0.0,0.00,1,0
970,1,CASH_OUT,1277212.77,C467632528,1277212.77,0.0,C716083600,0.0,2444985.19,1,0
1115,1,TRANSFER,35063.63,C1364127192,35063.63,0.0,C1136419747,0.0,0.00,1,0
